# Project 02 — NHS Regional Health Outcomes
## Phase 1 | Era 1 | Difficulty: Medium | DDaT: HEO Level

**Portfolio:** insightful-algorithms  
**Dataset:** ONS Health State Life Expectancies + OHID Fingertips Local Authority Health Profiles  
**Behaviour:** Communicating and Influencing (Civil Service HEO/SEO)  
**Objective:** Analyse regional variation in health outcomes across English local authorities, 
identify the relationship between deprivation and life expectancy, and communicate 
findings as a structured policy briefing for a non-technical public health audience.

## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Data Loading & Profiling](#2-data-loading--profiling)
3. [Data Cleaning & Preparation](#3-data-cleaning--preparation)
4. [Univariate Analysis](#4-univariate-analysis)
5. [Bivariate Analysis — Deprivation vs Health Outcomes](#5-bivariate-analysis)
6. [GroupBy Analysis — Regional Patterns](#6-groupby-analysis)
7. [Statistical Testing — North vs South](#7-statistical-testing)
8. [Key Findings & Policy Briefing](#8-key-findings--policy-briefing)

## 1. Environment Setup
Verify working directory, import libraries, confirm versions.

In [1]:
import os
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats

# Confirm working directory — must be project root
print("Working directory:", os.getcwd())

# Library versions
print(f"pandas:     {pd.__version__}")
print(f"numpy:      {np.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"seaborn:    {sns.__version__}")
print(f"scipy:      {stats.__version__}")

Working directory: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project2\notebooks
pandas:     3.0.1
numpy:      2.4.3
matplotlib: 3.10.8
seaborn:    0.13.2


AttributeError: module 'scipy.stats' has no attribute '__version__'

## 2. Data Loading & Profiling

We load all downloaded datasets and profile each one before selecting 
which to use for analysis. The `profile_dataframe()` function is carried 
forward from Project 01 as a reusable utility.

In [ ]:
def profile_dataframe(df, name="DataFrame"):
    """
    Profile a DataFrame: shape, dtypes, missing values, and basic stats.
    
    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame to profile.
    name : str
        Label to display in the output header.
    
    Returns
    -------
    None — prints profile to stdout.
    """
    print(f"\n{'='*60}")
    print(f"PROFILE: {name}")
    print(f"{'='*60}")
    print(f"Shape:   {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nColumn names:\n{df.columns.tolist()}")
    print(f"\nData types:\n{df.dtypes}")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Missing %': missing_pct
    })
    print(f"\nMissing values:\n{missing_df[missing_df['Missing Count'] > 0]}")
    print(f"\nFirst 3 rows:\n{df.head(3).to_string()}")
    print(f"\nBasic stats:\n{df.describe(include='all').to_string()}")

### 2.1 Load All Raw Files

We load all seven downloaded files and profile each one to understand 
their structure before deciding which to use for analysis.

In [ ]:
raw = r'data\raw'  # relative path from project root

# Load all files
files = {
    'le_single_year': ('lifeexpectancysingleyearperiods.xlsx', 'xlsx'),
    'le_local_areas': ('lifeexpectancylocalareas.xlsx', 'xlsx'),
    'hle_uk': ('healthylifeexpectancyuk.xlsx', 'xlsx'),
    'la_metadata': ('LocalAuthorityHealthProfiles.metadata.csv', 'csv'),
    'indicators_1': ('indicators-England.data.csv', 'csv'),
    'indicators_2': ('indicators-England.data (1).csv', 'csv'),
    'all_indicators_meta': ('Allindicators.metadata.csv', 'csv'),
}

dfs = {}

for key, (fname, ftype) in files.items():
    fpath = os.path.join(raw, fname)
    try:
        if ftype == 'xlsx':
            xl = pd.ExcelFile(fpath)
            print(f"\n{fname} — sheets: {xl.sheet_names}")
            dfs[key] = pd.read_excel(fpath, sheet_name=0)
        else:
            dfs[key] = pd.read_csv(fpath, encoding='utf-8', low_memory=False)
        print(f"Loaded '{key}': {dfs[key].shape}")
    except Exception as e:
        print(f"ERROR loading {fname}: {e}")

### 2.2 Profile Each Dataset

In [ ]:
for key, df in dfs.items():
    profile_dataframe(df, name=key)